# Multi-Objective Portfolio Optimization with cuOpt Python API

This notebook demonstrates how to use the cuOpt Python API to trace the **efficient frontier** of a mean-variance portfolio — the multi-objective core of portfolio optimization, where there's no single best balance of return and risk, only a curve of optimal tradeoffs.

The base `QP_portfolio_optimization` notebook solves for individual portfolios; here we sweep the whole frontier with the **ε-constraint method** and read each point's **sensitivity** (the return-constraint dual):

1. **Two objectives** — maximize return, minimize variance.
2. **Keep one as the objective, constrain the other** — minimize variance subject to `return ≥ ε`.
3. **Sweep ε** across the achievable return range; each solve is one frontier point.
4. **Read the frontier** — and, because this is a continuous QP, read each point's **dual**: the sensitivity d(variance)/d(return), how much extra variance one more unit of return costs.

A useful thing to notice: the base notebook already loops over target returns, so it is **already doing this** — naming the method is what lets you add the dual and reuse the recipe on any two-objective problem. (This workflow is also packaged as the `cuopt-multi-objective-exploration` skill.)

## Environment Setup

In [ ]:
import subprocess
import html
from IPython.display import display, HTML

def check_gpu():
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=5)
        result.check_returncode()
        lines = result.stdout.splitlines()
        gpu_info = lines[2] if len(lines) > 2 else "GPU detected"
        gpu_info_escaped = html.escape(gpu_info)
        display(HTML(f"""
        <div style="border:2px solid #4CAF50;padding:10px;border-radius:10px;background:#e8f5e9;">
            <h3>✅ GPU is enabled</h3>
            <pre>{gpu_info_escaped}</pre>
        </div>
        """))
        return True
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, FileNotFoundError, IndexError) as e:
        display(HTML("""
        <div style="border:2px solid red;padding:15px;border-radius:10px;background:#ffeeee;">
            <h3>⚠️ GPU not detected!</h3>
            <p>This notebook requires a <b>GPU runtime</b>.</p>

            <h4>If running in Google Colab:</h4>
            <ol>
              <li>Click on <b>Runtime → Change runtime type</b></li>
              <li>Set <b>Hardware accelerator</b> to <b>GPU</b></li>
              <li>Then click <b>Save</b> and <b>Runtime → Restart runtime</b>.</li>
            </ol>

            <h4>If running in Docker:</h4>
            <ol>
              <li>Ensure you have <b>NVIDIA Docker runtime</b> installed (<code>nvidia-docker2</code>)</li>
              <li>Run container with GPU support: <code>docker run --gpus all ...</code></li>
              <li>Or use: <code>docker run --runtime=nvidia ...</code> for older Docker versions</li>
              <li>Verify GPU access: <code>docker run --gpus all nvidia/cuda:12.0.0-base-ubuntu22.04 nvidia-smi</code></li>
            </ol>

            <p><b>Additional resources:</b></p>
            <ul>
              <li><a href="https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/install-guide.html" target="_blank">NVIDIA Container Toolkit Installation Guide</a></li>
            </ul>
        </div>
        """))
        return False

check_gpu()

In [ ]:
# Uncomment for your CUDA version if cuOpt is not already installed (e.g., Google Colab):
# !pip install --upgrade --extra-index-url https://pypi.nvidia.com cuopt-cu12  # CUDA 12
# !pip install --upgrade --extra-index-url https://pypi.nvidia.com cuopt-cu13  # CUDA 13

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cuopt.linear_programming.problem import Problem, QuadraticExpression, MINIMIZE
print("Imports ready (cuOpt QP solver)")

## Step 1 — the assets and their two objectives

Same simulated asset universe as `QP_portfolio_optimization`. The annualized **mean return** (to maximize) and **variance** / risk (to minimize) are the two competing objectives we trade off below.

In [ ]:
# Simulate monthly returns with realistic assumptions
np.random.seed(7)

assets = ["Cash", "US Equity", "Intl Equity", "Bond", "REIT/Gold"]

annual_mean = np.array([0.02, 0.08, 0.075, 0.04, 0.06])
annual_vol = np.array([0.005, 0.16, 0.18, 0.06, 0.14])

corr = np.array([
    [1.00, 0.05, 0.05, 0.10, 0.05],
    [0.05, 1.00, 0.80, -0.10, 0.55],
    [0.05, 0.80, 1.00, -0.05, 0.50],
    [0.10, -0.10, -0.05, 1.00, 0.00],
    [0.05, 0.55, 0.50, 0.00, 1.00],
])

monthly_mean = annual_mean / 12.0
monthly_vol = annual_vol / np.sqrt(12.0)
monthly_cov = np.outer(monthly_vol, monthly_vol) * corr

n_months = 120
returns = np.random.multivariate_normal(monthly_mean, monthly_cov, size=n_months)

# Estimate annualized mean and covariance from the simulated data
mean_returns = returns.mean(axis=0) * 12.0
cov_matrix = np.cov(returns, rowvar=False) * 12.0

summary = pd.DataFrame(
    {
        "Annualized Return": mean_returns,
        "Annualized Volatility": np.sqrt(np.diag(cov_matrix)),
    },
    index=assets,
)

summary.style.format({"Annualized Return": "{:.2%}", "Annualized Volatility": "{:.2%}"})

## Step 2 — minimize variance, capturing the return-constraint dual

The base notebook's `solve_min_variance_qp`, with one addition: we keep a handle on the `min_return` constraint and capture its **`.DualValue`** after the solve. For a continuous QP, that dual is the **sensitivity** d(variance)/d(return) — the marginal variance cost of demanding one more unit of return.

In [ ]:
def solve_min_variance_qp_dual(cov_matrix, mean_returns, target_return=None, max_weight=None):
    """Solve the min-variance QP and return the weights plus the return-constraint dual.

    Minimizes portfolio variance (w' * cov_matrix * w) subject to fully-invested
    weights and, when target_return is given, a minimum-return epsilon-constraint.
    The min_return constraint's .DualValue is the sensitivity d(variance)/d(return),
    accurate to cuOpt's barrier-solver tolerance (1e-8 by default).

    Parameters
    ----------
    cov_matrix : ndarray (n, n)
        Annualized covariance matrix (positive semidefinite).
    mean_returns : ndarray (n,)
        Annualized expected returns.
    target_return : float, optional
        Minimum portfolio return (the swept epsilon-constraint); None = unconstrained.
    max_weight : float, optional
        Upper bound on each asset weight (default 1.0).

    Returns
    -------
    dict
        {"weights", "ret", "vol", "dual", "status"}.
    """
    n = len(mean_returns)
    prob = Problem("Portfolio_Optimization")
    ub = max_weight if max_weight is not None else 1.0
    w = [prob.addVariable(lb=0.0, ub=ub, name=f"w_{i}") for i in range(n)]

    quad = None
    for i in range(n):
        for j in range(n):
            c = float(cov_matrix[i, j])
            if abs(c) > 1e-12:
                term = c * w[i] * w[j]
                quad = term if quad is None else quad + term
    prob.setObjective(quad, sense=MINIMIZE)

    prob.addConstraint(sum(w) == 1, name="fully_invested")
    ret_con = None
    if target_return is not None:
        ret_expr = sum(float(mean_returns[i]) * w[i] for i in range(n))
        ret_con = prob.addConstraint(ret_expr >= float(target_return), name="min_return")

    prob.solve()
    status = prob.Status.name if hasattr(prob.Status, "name") else str(prob.Status)
    weights = np.array([w[i].Value for i in range(n)])
    port_ret = float(mean_returns @ weights)
    port_vol = float(np.sqrt(max(weights @ cov_matrix @ weights, 0.0)))
    dual = abs(float(ret_con.DualValue)) if ret_con is not None else 0.0   # sensitivity d(var)/d(return)
    return {"weights": weights, "ret": port_ret, "vol": port_vol, "dual": dual, "status": status}

mv = solve_min_variance_qp_dual(cov_matrix, mean_returns)
print(f"Min-variance: status={mv['status']}, return={mv['ret']:.2%}, vol={mv['vol']:.2%}")

## Step 3 — sweep the return floor → the frontier (and its duals)

Sweep the return floor ε across the achievable range; each `ε` is one standard cuOpt solve, and we capture the dual at each point.

In [ ]:
min_ret = mv["ret"]
max_ret = float(mean_returns.max())
targets = np.linspace(min_ret, max_ret * 0.999, 25)

rets, vols, duals, flagged = [], [], [], 0
for t in targets:
    r = solve_min_variance_qp_dual(cov_matrix, mean_returns, target_return=t)
    if r["status"] not in ("Optimal", "PrimalFeasible"):
        continue
    if r["status"] != "Optimal":
        flagged += 1
    rets.append(r["ret"]); vols.append(r["vol"]); duals.append(r["dual"])

rets, vols, duals = map(np.array, (rets, vols, duals))
print(f"Frontier points: {len(rets)} | not certified-Optimal (PrimalFeasible): {flagged}")
print(f"Sensitivity d(variance)/d(return): {duals.min():.3f} -> {duals.max():.3f} as required return rises")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(vols * 100, rets * 100, "o-", color="navy", lw=1.6)
axes[0].set_xlabel("Volatility (%)"); axes[0].set_ylabel("Expected Return (%)")
axes[0].set_title("Efficient frontier (return vs risk)"); axes[0].grid(alpha=0.3)

axes[1].plot(rets * 100, duals, "o-", color="purple", lw=1.6)
axes[1].set_xlabel("Required return (%)"); axes[1].set_ylabel("Sensitivity  d(variance)/d(return)")
axes[1].set_title("Marginal risk cost of return (cuOpt QP dual)"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Step 4 — read the frontier

- The **frontier** (left) is the return-vs-risk Pareto set — every point is a min-variance portfolio for its return floor. There's no single "best"; you choose where on the curve to sit.
- The **dual** (right) is the **sensitivity** d(variance)/d(return): how much extra variance each additional unit of return costs. It rises along the frontier — the marginal cost of return steepens, which is exactly where a knee analysis pays off.

### Takeaway — reusing this on your own problem
Two competing objectives and a solver for one of them is all you need: keep one objective, turn the other into a swept constraint (`f₂ ≥ ε` or `≤ ε`), solve across the range, and read the frontier. If your code already loops over a target value, that loop **is** an ε-constraint sweep — name it, collect the non-dominated points, and (for an LP or QP) read the constraint's dual for the marginal exchange rate.

## License

SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.